In [1]:
import os
import numpy as np
from PIL import Image
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, f1_score, classification_report, confusion_matrix)
from sklearn.model_selection import GridSearchCV

In [2]:
IMAGE_SIZE = 128

train_dir = "Data/Training"
test_dir = "Data/Testing"

In [3]:
def load_images_from_folder(root_dir, image_size=128):
    X, y = [], []
    class_names = sorted(os.listdir(root_dir))
    class_to_index = {cls: i for i, cls in enumerate(class_names)}

    for cls in class_names:
        class_folder = os.path.join(root_dir, cls)
        if not os.path.isdir(class_folder):
            continue
        for file_name in os.listdir(class_folder):
            file_path = os.path.join(class_folder, file_name)
            try:
                image = Image.open(file_path).convert("L")
                image = image.resize((image_size, image_size))
                array = np.array(image, dtype=np.float32) / 255.0
                X.append(array.flatten())
                y.append(class_to_index[cls])
            except Exception:
                pass

    return np.stack(X), np.array(y), class_names

In [4]:
X_train, y_train, class_names = load_images_from_folder(train_dir, IMAGE_SIZE)
X_test, y_test, class_names = load_images_from_folder(test_dir, IMAGE_SIZE)

X_train.shape, X_test.shape, class_names

((5600, 16384),
 (1600, 16384),
 ['glioma', 'meningioma', 'notumor', 'pituitary'])

In [5]:
svm_classifier = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(kernel="rbf", C=1.0, gamma="scale"))
])

svm_classifier.fit(X_train, y_train)

y_pred = svm_classifier.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="macro")
cm = confusion_matrix(y_test, y_pred)

print(f"Baseline SVM accuracy: {accuracy:.4f}")
print(f"Baseline SVM macro F1: {f1:.4f}")
print(classification_report(y_test, y_pred, target_names=class_names))
print("Confusion matrix:\n", cm)

Baseline SVM accuracy: 0.8356
Baseline SVM macro F1: 0.8319
              precision    recall  f1-score   support

      glioma       0.82      0.68      0.74       400
  meningioma       0.79      0.77      0.78       400
     notumor       0.82      0.97      0.89       400
   pituitary       0.91      0.92      0.92       400

    accuracy                           0.84      1600
   macro avg       0.84      0.84      0.83      1600
weighted avg       0.84      0.84      0.83      1600

Confusion matrix:
 [[271  59  56  14]
 [ 42 307  29  22]
 [  5   5 390   0]
 [ 13  16   2 369]]
